In [1]:
import pandas as pd

df = pd.read_csv('../data/data.csv')

documents = df.to_dict(orient='records')

In [2]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [3]:
data_gen_instructions = """
You emulate a real user of a natural-remedy consultant application.

You will receive one record from the application's knowledge base.

Generate exactly 5 questions that a real user might ask and that can be
answered using the information in this record.

The questions will be used as ground-truth queries for evaluating a
retrieval system, so follow these rules carefully:

1. Every question must be answerable from the provided record.
2. Make the questions specific enough that this record is a relevant
   retrieval result.
3. Do not invent information that is not present in the record.
4. Write questions the way normal people might ask them online or in
   a chat application.
5. Keep each question complete and understandable on its own.
6. Do not make the questions overly formal, overly short, or unnecessarily long.
7. Paraphrase the record instead of copying its wording.
   Use synonyms and natural language where appropriate.
8. Do not simply turn field names into questions.
9. Make the 5 questions meaningfully different from one another.
10. Do not ask for an exact treatment dose.
11. Do not ask questions about record IDs, herb IDs, source URLs,
    last-reviewed dates, dataset flags, retrieval_text, or other
    implementation metadata.

Adapt the questions to the record_type:

- If record_type is "use_case":
  focus primarily on the condition, symptoms, possible remedy,
  traditional use, and modern evidence.

- If record_type is "herb_profile":
  focus on what the herb is, what it is traditionally used for,
  its general evidence, properties, or important safety information.

- If record_type is "preparation":
  focus on how the herb is prepared or used, differences between
  preparation forms, practical use, or how to buy an appropriate product.

- If record_type is "safety_interaction":
  focus on adverse effects, contraindications, medication interactions,
  pregnancy or surgery cautions, and when self-use may be inappropriate.

Use the herb name in some questions when natural, but do not require
every question to contain the herb name. For use-case records in particular,
include some symptom- or condition-based questions such as
"What might help with nausea?" so the retrieval system is tested on
realistic discovery queries.

Return only the 5 generated questions in the required structured format.
""".strip()

In [5]:
from dotenv import load_dotenv
from evaluation_utils import llm_structured_retry
from openai import OpenAI
import json
    
load_dotenv()
openai_client = OpenAI()

def generate_ground_truth(doc):
    user_prompt = json.dumps(
        doc,
        ensure_ascii=False,
        indent=2
    )

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "record_id": doc["record_id"],
            "herb_id": doc["herb_id"],
            "record_type": doc["record_type"]
        })

    return results, usage

In [6]:
from tqdm.auto import tqdm

ground_truths = []
usages = []

for doc in tqdm(documents):
    results, usage = generate_ground_truth(doc)
    ground_truths.extend(results)
    usages.append(usage)

  0%|          | 0/500 [00:00<?, ?it/s]

In [9]:
df_results = pd.DataFrame(ground_truths, columns=["record_id", "herb_id", "record_type", "question"])

In [15]:
df_results.to_csv('../data/ground_truths.csv', index=False)

In [18]:
## Retrieval Evaluation

In [25]:
df_ground_truth = pd.read_csv('../data/ground_truths.csv')

In [26]:
ground_truth = df_ground_truth.to_dict(orient='records')

In [35]:
len(ground_truth)

2500

In [41]:
import sys
print(sys.version)   # will say 3.9.6, not 3.12.13

3.9.6 (default, May 22 2026, 11:13:45) 
[Clang 21.0.0 (clang-2100.1.1.101)]


In [39]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

ModuleNotFoundError: No module named 'gitsource'

In [37]:
from minsearch import Index, VectorSearch
from embedder import Embedder

embed = Embedder()
import numpy as np

# Build Text Search
index = Index(
    text_fields=["content"]
)
index.fit(chunks)

# Build Vector Search
matrix = []

for chunk in chunks:
    vector = embed.encode_batch([chunk["content"]])
    matrix.append(vector)

X = np.vstack(matrix)

vector_index = VectorSearch(
    keyword_fields=["content"]
)

vector_index.fit(X, chunks)

ModuleNotFoundError: No module named 'minsearch'